In [ ]:

# IMPORTS
import pandas as pd
import ast
import re
import sys


# load shared ingredient cleaning functions
from ingredient_normalisation import (
    remove_unnecessary_words,
    remove_quantity_noise,
    singularise_ingredient,
    map_synonyms,
    normalise_user_ingredients
)

In [ ]:
# load recipe dataset
recipes_dataset = pd.read_csv("recipes_dataset.csv")
# use copy to create independent DF so original isn't altered
recipes_dataset_normalised = recipes_dataset.copy()

In [ ]:
# STEP 1
# convert stored ingredient strings back into lists
recipes_dataset_normalised["processing_ingredients"] = (
    recipes_dataset_normalised["processing_ingredients"]
    .apply(ast.literal_eval) # usde ast.literal_eval to convert strings to python objects
)


In [ ]:
# STEP 2
# convert every ingredient to lowercase for consistent matching
recipes_dataset_normalised["processing_ingredients"] = (
    recipes_dataset_normalised["processing_ingredients"]
    .apply(lambda ingredient_list: [ingredient.lower() for ingredient in ingredient_list]) # convert lowercase in list
)

In [ ]:
#STEP 3 
# remove whitespace from start and end of each ingredient
recipes_dataset_normalised["processing_ingredients"] = (
    recipes_dataset_normalised["processing_ingredients"]
    .apply(
        lambda ingredient_list: [
            ingredient.strip() for ingredient in ingredient_list # use the python strip function to remove whitespace
        ]
    )
)

In [ ]:
# STEP 4 - REMOVE UNNECESSARY ADDITIONAL WORDS

# remove preparation and descriptive words
recipes_dataset_normalised["processing_ingredients"] = (
    recipes_dataset_normalised["processing_ingredients"]
    .apply(
        lambda ingredient_list: [
            remove_unnecessary_words(ingredient)
            for ingredient in ingredient_list
        ]
    )
)

In [ ]:
# STEP 5 - REMOVE QUANTITY AND MEASUREMENT NOISE

# remove quantities, measurements and noise
recipes_dataset_normalised["processing_ingredients"] = (
    recipes_dataset_normalised["processing_ingredients"]
    .apply(
        lambda ingredient_list: [
            remove_quantity_noise(ingredient)
            for ingredient in ingredient_list
        ]
    )
)

In [ ]:
# STEP 6 - REMOVE PLURALS / STANDARDISE TO SINGULAR INGREDIENT NAMES

# convert plurals to singular ingredient names
recipes_dataset_normalised["processing_ingredients"] = (
    recipes_dataset_normalised["processing_ingredients"]
    .apply(lambda ingredient_list: [singularise_ingredient(ingredient)
            for ingredient in ingredient_list]
    )
)

In [ ]:
# STEP 7 - MAP SYNONYMS to one

# map synonyms to canonical ingredient names
recipes_dataset_normalised["processing_ingredients"] = (
    recipes_dataset_normalised["processing_ingredients"]
    .apply(lambda ingredient_list: [map_synonyms(ingredient) for ingredient in ingredient_list]
    )
)

In [ ]:
# STEP 8 

# remove empty ingredients left after cleaning
recipes_dataset_normalised["processing_ingredients"] = (
    recipes_dataset_normalised["processing_ingredients"]
    .apply(lambda ingredient_list: [ingredient for ingredient in ingredient_list 
                                    if ingredient.strip()]
    )
)

In [ ]:
# STEP 9 - REMOVE DUPLICATE INGREDIENTS WITHIN EACH RECIPE

# remove repeated ingredients and keep original ingredient order
recipes_dataset_normalised["processing_ingredients"] = (
    recipes_dataset_normalised["processing_ingredients"]
    .apply(lambda ingredient_list: list(dict.fromkeys(ingredient_list)))
)

In [ ]:
# STEP 10 - save

# save normalised dataset without adding DF index column
recipes_dataset_normalised.to_csv(
    "recipes_dataset_normalised.csv",
    index=False
)